# Amazon Bedrock — Distributed Inference Engine (Project Mantle)

This notebook walks through all the tutorials in the **Project Mantle** module of the
Amazon Bedrock Workshop. Project Mantle powers the OpenAI API-compatible endpoints
(Chat Completions and Responses API), enabling you to use the familiar OpenAI SDK
with Amazon Bedrock models.

**What you'll learn:**
1. Setup & Authentication
2. Discover Models
3. Chat Completions API
4. Responses API
5. Tool Use (Function Calling)
6. Structured Output
7. Background Mode
8. Guardrails
9. Projects API

**Prerequisites:**
- An AWS account with access to Amazon Bedrock
- Python 3.9+
- A Bedrock API key or AWS credentials configured

**Estimated time:** ~90 minutes

---
## 1. Setup & Authentication

Install the required dependencies and configure your environment.

In [ ]:
# Install the OpenAI Python SDK (and optional SigV4 token generator)
%pip install --quiet openai requests

### Option A: Bedrock API Key (Recommended for this workshop)

1. Open the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home)
2. In the left navigation, select **API keys**
3. Generate a short-term or long-term API key
4. Set the environment variables below

In [ ]:
import os

# Set your credentials here (or export them in your terminal before launching Jupyter)
# Replace with your actual values
os.environ["OPENAI_BASE_URL"] = "https://bedrock-mantle.us-east-1.api.aws/v1"
os.environ["OPENAI_API_KEY"] = "your-bedrock-api-key-here"  # Replace with your key

### Option B: SigV4 Authentication (AWS Credentials)

If you already have AWS credentials configured (e.g., via `aws configure`, IAM roles,
or environment variables), uncomment and run the cell below instead of Option A.

In [ ]:
# # Uncomment to use SigV4 authentication
%pip install --quiet aws-bedrock-token-generator
#
# from aws_bedrock_token_generator import provide_token
# import os
#
# os.environ["OPENAI_BASE_URL"] = "https://bedrock-mantle.us-east-1.api.aws/v1"
# os.environ["OPENAI_API_KEY"] = provide_token()

### Verify the Setup

In [ ]:
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_BASE_URL and OPENAI_API_KEY from env

# Quick test: list available models
models = client.models.list()
print(f"✅ Connected! Found {len(models.data)} models.")
for m in models.data[:5]:
    print(f"   - {m.id}")

---
## 2. Discover Models

The Models API lets you discover which models are available in your region.

In [ ]:
# List all available models on Amazon Bedrock via the Mantle endpoint
from openai import OpenAI

client = OpenAI()

models = client.models.list()

print(f"Found {len(models.data)} models:\n")
for model in models.data:
    print(f"  - {model.id}  (owned by: {model.owned_by})")

In [ ]:
# Retrieve a specific model by ID
model = client.models.retrieve("openai.gpt-oss-120b")
print(f"Model ID: {model.id}")
print(f"Owned by: {model.owned_by}")
print(f"Created:  {model.created}")

---
## 3. Chat Completions API

The Chat Completions API is the stateless, familiar pattern. You send the full
conversation history with each request and get a response back.

### Basic Chat Completion

In [ ]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful cloud architecture assistant."
        },
        {
            "role": "user",
            "content": "What are the key benefits of serverless architectures?"
        }
    ]
)

print(completion.choices[0].message.content)

### Streaming

In [ ]:
# Stream chat completion responses in real-time
from openai import OpenAI

client = OpenAI()

stream = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a concise technical writer."},
        {"role": "user", "content": "Explain the CAP theorem in simple terms."}
    ],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()

### Multi-turn Conversation (Client-managed)

In [ ]:
# Multi-turn conversation with client-managed history
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "system", "content": "You are a helpful AWS solutions architect."}
]

def chat(user_message: str) -> str:
    """Send a message and get a response, maintaining conversation history."""
    messages.append({"role": "user", "content": user_message})

    completion = client.chat.completions.create(
        model="openai.gpt-oss-120b",
        messages=messages
    )

    assistant_message = completion.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})
    return assistant_message

# Turn 1
print("User: I need to build a real-time data pipeline.")
print(f"Assistant: {chat('I need to build a real-time data pipeline.')}\n")

# Turn 2
print("User: What AWS services would you recommend?")
print(f"Assistant: {chat('What AWS services would you recommend?')}\n")

# Turn 3
print("User: How would I handle failures in this pipeline?")
print(f"Assistant: {chat('How would I handle failures in this pipeline?')}")

---
## 4. Responses API

The Responses API manages conversation state on the server side. You chain responses
using `previous_response_id` instead of sending the full history each time.

### Basic Response

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[
        {"role": "user", "content": "What is Amazon Bedrock?"}
    ]
)

print(response.output_text)

### Stateful Multi-turn Conversation

In [ ]:
# Stateful multi-turn conversation — the server maintains context
from openai import OpenAI

client = OpenAI()

# Turn 1
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "What is the capital of France?"}]
)
print(f"Turn 1: {response.output_text}")
print(f"Response ID: {response.id}\n")

# Turn 2 — chain using previous_response_id
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "What river runs through it?"}],
    previous_response_id=response.id
)
print(f"Turn 2: {response.output_text}")
print(f"Response ID: {response.id}\n")

# Turn 3 — continue the chain
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "How long is that river?"}],
    previous_response_id=response.id
)
print(f"Turn 3: {response.output_text}")

### Streaming with the Responses API

In [ ]:
from openai import OpenAI

client = OpenAI()

stream = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "Write a haiku about cloud computing."}],
    stream=True
)

for event in stream:
    if hasattr(event, 'type') and event.type == 'response.output_text.delta':
        print(event.delta, end="", flush=True)

print()

### Retrieve a Previous Response

In [ ]:
from openai import OpenAI

client = OpenAI()

# Create a response
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "Give me 3 tips for writing clean code."}]
)

response_id = response.id
print(f"Response ID: {response_id}\n")

# Later, retrieve it by ID
retrieved = client.responses.retrieve(response_id)
print(f"Retrieved: {retrieved.output_text}")

---
## 5. Tool Use (Function Calling)

Tool use allows the model to call functions you define, enabling it to interact
with external systems. The model tells you **which** tool to call and **with what
arguments**, and you send the result back.

### Define a Sample Tool

In [ ]:
import json

def get_weather(city: str, unit: str = "celsius") -> str:
    """Simulated weather lookup — replace with a real API call in production."""
    weather_data = {
        "Seattle": {"temp": 12, "condition": "Rainy"},
        "San Francisco": {"temp": 18, "condition": "Foggy"},
        "New York": {"temp": 25, "condition": "Sunny"},
        "Tokyo": {"temp": 28, "condition": "Humid"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    return json.dumps({
        "city": city,
        "temperature": data["temp"],
        "unit": unit,
        "condition": data["condition"]
    })

### Tool Use with the Responses API

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

# Define the tool schema
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name, e.g. Seattle"},
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit"
                }
            },
            "required": ["city"]
        }
    }
]

# Step 1: Send the request
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[{"role": "user", "content": "What's the weather like in Seattle?"}],
    tools=tools
)

# Step 2: Check if the model wants to call a tool
for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"Tool called: {item.name}({args})")
        print(f"Tool result: {result}\n")

        # Step 3: Send the tool result back
        final_response = client.responses.create(
            model="openai.gpt-oss-120b",
            previous_response_id=response.id,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": result
                }
            ]
        )
        print(f"Assistant: {final_response.output_text}")

### Tool Use with the Chat Completions API

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

# Tool definition for Chat Completions (note the nested "function" key)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"},
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Step 1: Send the request
messages = [{"role": "user", "content": "What's the weather in New York?"}]

completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=messages,
    tools=tools
)

message = completion.choices[0].message

# Step 2: If the model wants to call a tool, execute it
if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    result = get_weather(**args)
    print(f"Tool called: {tool_call.function.name}({args})")
    print(f"Tool result: {result}\n")

    # Step 3: Send the tool result back (include full history)
    messages.append(message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })

    followup = client.chat.completions.create(
        model="openai.gpt-oss-120b",
        messages=messages
    )
    print(f"Assistant: {followup.choices[0].message.content}")

---
## 6. Structured Output

Structured outputs ensure model responses conform to a JSON schema you define.
The model is **guaranteed** to return valid, schema-compliant JSON.

### JSON Schema Output Format

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a data extraction assistant. Extract structured data from the user's text."
        },
        {
            "role": "user",
            "content": (
                "Our Q3 revenue was $4.2 billion, up 15% year-over-year. "
                "Operating income reached $800 million with a margin of 19%. "
                "We added 2,500 new enterprise customers this quarter."
            )
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "financial_summary",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "revenue": {"type": "string", "description": "Total revenue amount"},
                    "revenue_growth": {"type": "string", "description": "YoY revenue growth %"},
                    "operating_income": {"type": "string", "description": "Operating income amount"},
                    "operating_margin": {"type": "string", "description": "Operating margin %"},
                    "new_customers": {"type": "integer", "description": "Number of new enterprise customers"}
                },
                "required": ["revenue", "revenue_growth", "operating_income", "operating_margin", "new_customers"],
                "additionalProperties": False
            }
        }
    }
)

result = json.loads(completion.choices[0].message.content)
print(json.dumps(result, indent=2))

### Strict Tool Use

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "create_support_ticket",
            "description": "Create a customer support ticket from the user's message",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Short title for the ticket"},
                    "priority": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"],
                        "description": "Ticket priority level"
                    },
                    "category": {
                        "type": "string",
                        "enum": ["billing", "technical", "account", "general"],
                        "description": "Ticket category"
                    },
                    "description": {"type": "string", "description": "Detailed description of the issue"}
                },
                "required": ["title", "priority", "category", "description"],
                "additionalProperties": False
            }
        }
    }
]

completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": (
                "I've been charged twice for my last invoice and I need "
                "this resolved urgently. My account ID is 12345."
            )
        }
    ],
    tools=tools,
    tool_choice="auto"
)

if completion.choices[0].message.tool_calls:
    tool_call = completion.choices[0].message.tool_calls[0]
    ticket = json.loads(tool_call.function.arguments)
    print("Support ticket created:")
    print(json.dumps(ticket, indent=2))

---
## 7. Background Mode

Background mode enables asynchronous inference for long-running tasks. You submit
a request, receive a response ID immediately, and poll for completion.

> **Note:** Background mode is only available with the Responses API.

In [ ]:
# Background mode: Asynchronous inference for long-running tasks
from openai import OpenAI
import time

client = OpenAI()

# Submit the request in background mode
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[
        {
            "role": "user",
            "content": (
                "Write a detailed technical comparison of microservices vs "
                "monolithic architectures, covering scalability, deployment, "
                "testing, team organization, and operational complexity. "
                "Include specific examples and trade-offs for each approach."
            )
        }
    ],
    background=True
)

print(f"Request submitted. Response ID: {response.id}")
print(f"Initial status: {response.status}")

# Poll for completion
while response.status == "in_progress":
    print("  ⏳ Still processing...")
    time.sleep(2)
    response = client.responses.retrieve(response.id)

print(f"\nFinal status: {response.status}")
print(f"\nResponse:\n{response.output_text}")

### Background Mode with Stateful Conversations

In [ ]:
from openai import OpenAI
import time

client = OpenAI()

def background_chat(client, model, user_input, previous_id=None):
    """Submit a background request and wait for completion."""
    response = client.responses.create(
        model=model,
        input=[{"role": "user", "content": user_input}],
        previous_response_id=previous_id,
        background=True
    )
    print(f"  Submitted (ID: {response.id})")

    while response.status == "in_progress":
        time.sleep(1)
        response = client.responses.retrieve(response.id)

    return response

MODEL = "openai.gpt-oss-120b"

# Turn 1
print("User: Analyze the pros and cons of event-driven architecture.")
resp = background_chat(client, MODEL, "Analyze the pros and cons of event-driven architecture.")
print(f"Assistant: {resp.output_text}\n")

# Turn 2 — chained via previous_response_id
print("User: Now compare it with request-response patterns for a payment system.")
resp = background_chat(client, MODEL,
    "Now compare it with request-response patterns for a payment system.",
    previous_id=resp.id
)
print(f"Assistant: {resp.output_text}")

---
## 8. Guardrails

Amazon Bedrock Guardrails help you implement safeguards for your generative AI
applications. You can apply guardrails to evaluate both user inputs and model responses.

**Before running this section**, create a guardrail in the
[Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/guardrails)
and note the Guardrail ID and Version.

In [ ]:
# Using Bedrock Guardrails with the Chat Completions API
from openai import OpenAI

client = OpenAI()

# Replace with your guardrail ID and version from the Bedrock console
GUARDRAIL_ID = "your-guardrail-id"
GUARDRAIL_VERSION = "DRAFT"

completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful customer service assistant."
        },
        {
            "role": "user",
            "content": "Tell me about your product return policy."
        }
    ],
    extra_body={
        "guardrailConfig": {
            "guardrailIdentifier": GUARDRAIL_ID,
            "guardrailVersion": GUARDRAIL_VERSION
        }
    }
)

print(completion.choices[0].message.content)

### Testing Guardrail Intervention

In [ ]:
# Test guardrail intervention with content that should be blocked
from openai import OpenAI

client = OpenAI()

GUARDRAIL_ID = "your-guardrail-id"
GUARDRAIL_VERSION = "DRAFT"

try:
    completion = client.chat.completions.create(
        model="openai.gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": "Tell me how to do something harmful."
            }
        ],
        extra_body={
            "guardrailConfig": {
                "guardrailIdentifier": GUARDRAIL_ID,
                "guardrailVersion": GUARDRAIL_VERSION
            }
        }
    )
    print(completion.choices[0].message.content)
except Exception as e:
    print(f"Guardrail intervened: {e}")

### Independent Guardrail Evaluation (ApplyGuardrail API)

Evaluate content against guardrails **without invoking a model**. This uses the
Bedrock Runtime SDK (boto3), not the OpenAI SDK.

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

GUARDRAIL_ID = "your-guardrail-id"
GUARDRAIL_VERSION = "DRAFT"

response = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION,
    source="INPUT",
    content=[
        {
            "text": {
                "text": "My email is user@example.com and my phone is 555-0123."
            }
        }
    ]
)

print(f"Action: {response['action']}")

if response['action'] == 'GUARDRAIL_INTERVENED':
    print(f"Blocked/Masked: {response['outputs'][0]['text']}")
else:
    print("Content passed guardrail evaluation.")

---
## 9. Projects API

The Projects API provides application-level isolation for your generative AI workloads.
Each project acts as a logical boundary — giving you separate access control, cost
tracking, and observability without managing multiple AWS accounts.

> **Note:** Projects can only be used with models on the bedrock-mantle endpoint.

### Create a Project

In [ ]:
import os
import requests
import json

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]

# Create a project with tags for cost tracking
response = requests.post(
    f"{base_url}/organization/projects",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    json={
        "name": "Workshop-Demo-Project",
        "tags": {
            "Project": "BedrockWorkshop",
            "Environment": "Development",
            "Owner": "WorkshopParticipant"
        }
    }
)

project = response.json()
print(f"Project created: {project['id']}")
print(f"ARN: {project['arn']}")
print(f"Status: {project['status']}")
print(f"Tags: {json.dumps(project.get('tags', {}), indent=2)}")

# Store the project ID for later use
PROJECT_ID = project["id"]

### List Projects

In [ ]:
import os
import requests
import json

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]

response = requests.get(
    f"{base_url}/organization/projects",
    headers={"Authorization": f"Bearer {api_key}"}
)

projects = response.json()
print(f"Found {len(projects['data'])} projects:\n")
for p in projects["data"]:
    print(f"  - {p['name']} (ID: {p['id']}, Status: {p['status']})")

### Use a Project for Inference

In [ ]:
from openai import OpenAI

# Pass the project ID when creating the client
# Replace PROJECT_ID with your actual project ID from the creation step above
client = OpenAI(project=PROJECT_ID)

# Responses API — request is associated with the project
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=[
        {"role": "user", "content": "What are the key features of Amazon Bedrock?"}
    ]
)
print(f"[Project: {PROJECT_ID}]")
print(response.output_text)

In [ ]:
# Chat Completions API — also associated with the project
completion = client.chat.completions.create(
    model="openai.gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Explain how projects improve security in Amazon Bedrock."}
    ]
)
print(f"\n[Project: {PROJECT_ID}]")
print(completion.choices[0].message.content)

### Archive a Project (Cleanup)

When you're done with a project, archive it. Archived projects cannot accept new
inference requests, but historical data remains accessible for up to 30 days.

In [ ]:
import os
import requests

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]

# Archive the workshop demo project
response = requests.post(
    f"{base_url}/organization/projects/{PROJECT_ID}/archive",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
)

if response.status_code == 200:
    print(f"✅ Project {PROJECT_ID} archived successfully.")
else:
    print(f"Archive response: {response.status_code} - {response.text}")

---
## 🎉 Workshop Complete!

You've now explored all the key capabilities of Amazon Bedrock's distributed inference
engine (Project Mantle) through the OpenAI-compatible APIs:

- **Setup & Authentication** — API keys and SigV4 credentials
- **Models API** — Discovering available models
- **Chat Completions** — Stateless text generation, streaming, multi-turn
- **Responses API** — Stateful conversations with server-managed context
- **Tool Use** — Function calling with both APIs
- **Structured Output** — Schema-compliant JSON responses
- **Background Mode** — Async inference for long-running tasks
- **Guardrails** — Content filtering and safety controls
- **Projects API** — Workload isolation, cost tracking, and access control

For more information, visit the
[Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/bedrock-mantle.html).